# Classical ML Benchmark — Wildfire Prediction

This notebook establishes a classical machine learning baseline for predicting wildfire occurrence (num_fires) per California zip code per month.  
Results serve as a comparison target for a future Quantum ML model.

**Dataset:** `features.csv` — 155,580 rows of California zip+month data, 2018–2022  
**Temporal split:** Train 2018–2020 | Validate 2021 | Test 2022  
**Target:** `num_fires` (regression; severe class imbalance — ~99% zeros)

---
## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBRegressor

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

DATA_DIR = '/Users/jacobgutsin/Documents/Deloitte QML Competition/'
print('Imports complete.')

---
## 2. Load Data & Class Distribution

In [ ]:
df = pd.read_csv(DATA_DIR + 'features.csv')

# Parse year from year_month (e.g. '2018-01' → 2018)
df['year_parsed'] = df['year_month'].str[:4].astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Year range: {df["year_parsed"].min()} – {df["year_parsed"].max()}')
print()
print(df.head(3).to_string())

In [ ]:
# --- Class distribution ---
total = len(df)
fire_rows    = (df['num_fires'] > 0).sum()
no_fire_rows = (df['num_fires'] == 0).sum()

print('=== Class Distribution (full dataset) ===')
print(f'  Total rows       : {total:>8,}')
print(f'  Rows with fires  : {fire_rows:>8,}  ({100*fire_rows/total:.2f}%)')
print(f'  Rows without fire: {no_fire_rows:>8,}  ({100*no_fire_rows/total:.2f}%)')
print()
print('num_fires value counts (top 10):')
print(df['num_fires'].value_counts().head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie
axes[0].pie(
    [no_fire_rows, fire_rows],
    labels=['No Fire (0)', 'Fire (>0)'],
    autopct='%1.2f%%',
    colors=['#4878CF', '#E24A33'],
    startangle=90
)
axes[0].set_title('Fire vs No-Fire (full dataset)')

# Histogram of num_fires > 0
fire_vals = df.loc[df['num_fires'] > 0, 'num_fires']
axes[1].hist(fire_vals, bins=40, color='#E24A33', edgecolor='white', linewidth=0.5)
axes[1].set_title('Distribution of num_fires (fire rows only)')
axes[1].set_xlabel('num_fires')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 3. Temporal Train / Validate / Test Split

In [ ]:
train_df = df[df['year_parsed'] <= 2020].copy()
val_df   = df[df['year_parsed'] == 2021].copy()
test_df  = df[df['year_parsed'] == 2022].copy()

print('=== Split Summary ===')
for name, split in [('Train (2018-2020)', train_df), ('Val   (2021)', val_df), ('Test  (2022)', test_df)]:
    n = len(split)
    f = (split['num_fires'] > 0).sum()
    print(f'  {name}: {n:>7,} rows  |  {f:>4,} fire rows ({100*f/n:.2f}%)')

---
## 4. Features & Sample Weights

In [ ]:
FEATURES = [
    'month_sin', 'month_cos', 'year',
    'avg_tmax_c', 'temp_range', 'tot_prcp_mm',
    'dryness_3m_avg', 'latitude', 'longitude'
]
TARGET = 'target'          # log1p(num_fires)

X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values

X_val   = val_df[FEATURES].values
y_val   = val_df[TARGET].values

X_test  = test_df[FEATURES].values
y_test  = test_df[TARGET].values

# Ground-truth num_fires (original scale) for evaluation
y_val_raw  = val_df['num_fires'].values
y_test_raw = test_df['num_fires'].values

# Sample weights: 10x for fire rows, 1x for no-fire rows
FIRE_WEIGHT = 10
sample_weights = np.where(train_df['num_fires'].values > 0, FIRE_WEIGHT, 1.0)

print(f'Feature count  : {len(FEATURES)}')
print(f'Features       : {FEATURES}')
print(f'Training rows  : {X_train.shape[0]:,}')
print(f'Fire-row weight: {FIRE_WEIGHT}x')
print(f'Weighted fire rows ≈ {sample_weights[sample_weights > 1].sum():,.0f} effective observations')

---
## 5. Train All Models & Evaluate on Validation Set

In [ ]:
def evaluate(y_true_log, y_pred_log, y_true_raw, threshold=0.5):
    """Compute regression and binary classification metrics.
    
    Parameters
    ----------
    y_true_log : array  — ground truth in log1p scale
    y_pred_log : array  — predictions in log1p scale
    y_true_raw : array  — ground truth num_fires (original scale)
    threshold  : float  — predicted num_fires > threshold → fire
    """
    # Convert back to original scale
    y_pred_raw = np.expm1(y_pred_log).clip(min=0)

    mae  = mean_absolute_error(y_true_raw, y_pred_raw)
    rmse = np.sqrt(mean_squared_error(y_true_raw, y_pred_raw))
    r2   = r2_score(y_true_raw, y_pred_raw)

    # Binary: actual fire if num_fires > 0
    y_bin_true = (y_true_raw > 0).astype(int)
    y_bin_pred = (y_pred_raw > threshold).astype(int)

    prec   = precision_score(y_bin_true, y_bin_pred, zero_division=0)
    recall = recall_score(y_bin_true, y_bin_pred, zero_division=0)
    f1     = f1_score(y_bin_true, y_bin_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_bin_true, y_pred_raw)
    except Exception:
        auc = float('nan')

    return dict(MAE=mae, RMSE=rmse, R2=r2,
                Precision=prec, Recall=recall, F1=f1, AUC_ROC=auc)

In [ ]:
# ---- Model definitions ----
models = {
    'Dummy (predict 0)': DummyRegressor(strategy='constant', constant=0),
    'Ridge Regression' : Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest'    : RandomForestRegressor(
                             n_estimators=300,
                             max_depth=12,
                             min_samples_leaf=5,
                             n_jobs=-1,
                             random_state=RANDOM_STATE
                         ),
    'Gradient Boosting': GradientBoostingRegressor(
                             n_estimators=300,
                             learning_rate=0.05,
                             max_depth=5,
                             subsample=0.8,
                             random_state=RANDOM_STATE
                         ),
    'XGBoost'          : XGBRegressor(
                             n_estimators=300,
                             learning_rate=0.05,
                             max_depth=6,
                             subsample=0.8,
                             colsample_bytree=0.8,
                             reg_lambda=1.0,
                             reg_alpha=0.1,
                             random_state=RANDOM_STATE,
                             n_jobs=-1,
                             verbosity=0
                         ),
}

# Models that accept sample_weight in fit()
SUPPORTS_SAMPLE_WEIGHT = {'Ridge Regression', 'Random Forest',
                          'Gradient Boosting', 'XGBoost'}

print('Models defined:')
for k in models:
    print(f'  • {k}')

In [ ]:
val_results  = {}
trained_models = {}

for name, model in models.items():
    print(f'Training: {name} ...', end=' ', flush=True)
    if name in SUPPORTS_SAMPLE_WEIGHT:
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)
    metrics = evaluate(y_val, y_pred_val, y_val_raw)
    val_results[name] = metrics
    trained_models[name] = model
    print(f'done  |  F1={metrics["F1"]:.4f}  AUC={metrics["AUC_ROC"]:.4f}')

print('\nAll models trained.')

In [ ]:
val_df_results = pd.DataFrame(val_results).T
val_df_results = val_df_results[['MAE','RMSE','R2','Precision','Recall','F1','AUC_ROC']]
val_df_results = val_df_results.round(4)

print('=== Validation Set Results (2021) ===')
print(val_df_results.to_string())

# Highlight best per metric
val_df_results.style \
    .highlight_min(subset=['MAE','RMSE'], color='#c6efce') \
    .highlight_max(subset=['R2','Precision','Recall','F1','AUC_ROC'], color='#c6efce') \
    .format(precision=4) \
    .set_caption('Validation Results — green = best per column')

In [ ]:
# Plot validation metrics as a grouped bar chart
metrics_to_plot = ['MAE', 'RMSE', 'F1', 'AUC_ROC']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

model_names  = list(val_results.keys())
colors = sns.color_palette('muted', len(model_names))

for ax, metric in zip(axes, metrics_to_plot):
    values = [val_results[m][metric] for m in model_names]
    bars = ax.barh(model_names, values, color=colors)
    ax.set_title(metric)
    ax.set_xlabel(metric)
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=8)

plt.suptitle('Model Comparison — Validation Set (2021)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Identify best model by AUC-ROC (robust for imbalanced data)
best_model_name = max(val_results, key=lambda m: val_results[m]['AUC_ROC'])
best_model = trained_models[best_model_name]
print(f'Best model (highest AUC-ROC on val): {best_model_name}')

---
## 6. Best Model — Full Evaluation on Test Set (2022)

In [ ]:
y_pred_test_log = best_model.predict(X_test)
y_pred_test_raw = np.expm1(y_pred_test_log).clip(min=0)

test_metrics = evaluate(y_test, y_pred_test_log, y_test_raw)

print(f'=== Test Set Results (2022) — {best_model_name} ===')
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

In [ ]:
# Validate vs test comparison
compare_df = pd.DataFrame({
    'Validation (2021)': val_results[best_model_name],
    'Test (2022)'      : test_metrics
}).round(4)
print(f'Val vs Test comparison — {best_model_name}')
print(compare_df.to_string())

In [ ]:
# Confusion matrix on test set
y_bin_true_test = (y_test_raw > 0).astype(int)
y_bin_pred_test = (y_pred_test_raw > 0.5).astype(int)

cm = confusion_matrix(y_bin_true_test, y_bin_pred_test)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Fire', 'Fire'])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_model_name}\n(Test Set 2022, threshold = 0.5 fires)')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
print(f'Fire recall (sensitivity): {tp/(tp+fn):.4f}')
print(f'No-fire specificity      : {tn/(tn+fp):.4f}')

In [ ]:
# Actual vs Predicted scatter (log scale)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: log1p scale
axes[0].scatter(y_test, y_pred_test_log, alpha=0.15, s=8, color='steelblue')
lim = max(y_test.max(), y_pred_test_log.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual target (log1p scale)')
axes[0].set_ylabel('Predicted target (log1p scale)')
axes[0].set_title('Actual vs Predicted (log1p)')
axes[0].legend()

# Scatter: original scale (fire rows only)
fire_mask = y_test_raw > 0
axes[1].scatter(y_test_raw[fire_mask], y_pred_test_raw[fire_mask],
                alpha=0.3, s=10, color='#E24A33')
lim2 = max(y_test_raw[fire_mask].max(), y_pred_test_raw[fire_mask].max()) * 1.05
axes[1].plot([0, lim2], [0, lim2], 'k--', linewidth=1.5, label='Perfect fit')
axes[1].set_xlabel('Actual num_fires')
axes[1].set_ylabel('Predicted num_fires')
axes[1].set_title('Actual vs Predicted — num_fires (fire rows only)')
axes[1].legend()

plt.suptitle(f'{best_model_name} — Test Set 2022', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Feature Importance (Tree Models)

In [ ]:
tree_model_names = ['Random Forest', 'Gradient Boosting', 'XGBoost']
n_tree = len(tree_model_names)

fig, axes = plt.subplots(1, n_tree, figsize=(6 * n_tree, 5))

for ax, name in zip(axes, tree_model_names):
    model = trained_models[name]
    importances = model.feature_importances_
    sorted_idx  = np.argsort(importances)

    ax.barh(
        [FEATURES[i] for i in sorted_idx],
        importances[sorted_idx],
        color=sns.color_palette('viridis', len(FEATURES))
    )
    ax.set_title(f'{name}\nFeature Importances')
    ax.set_xlabel('Importance')
    ax.set_xlim(0, importances.max() * 1.2)

plt.tight_layout()
plt.suptitle('Feature Importances — Tree Models', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Heatmap comparing feature importances across tree models
importance_df = pd.DataFrame(
    {name: trained_models[name].feature_importances_ for name in tree_model_names},
    index=FEATURES
)
# Normalise each column to sum to 1 (already the case for sklearn trees, but ensures consistency)
importance_df = importance_df.div(importance_df.sum(axis=0), axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(importance_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Normalised Importance'})
ax.set_title('Feature Importance Heatmap — Tree Models')
ax.set_xlabel('Model')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

print('\nFeature importance table:')
print(importance_df.round(4).to_string())

---
## 8. Generate 2023 Predictions

In [ ]:
# ── 8a. Build historical zip+month feature averages from training data ──────
# We use ALL available data (2018-2022) for historical averages
historical_features = [
    'avg_tmax_c', 'temp_range', 'tot_prcp_mm',
    'dryness_3m_avg', 'latitude', 'longitude'
]

# Parse integer month from year_month
df['month_int'] = df['year_month'].str[5:7].astype(int)

# Group by zip + month, take mean of each feature
zip_month_avgs = (
    df.groupby(['zip', 'month_int'])[historical_features]
    .mean()
    .reset_index()
)
print(f'zip+month combinations with historical data: {len(zip_month_avgs):,}')
print(zip_month_avgs.head(3).to_string())

In [ ]:
# ── 8b. Build synthetic 2023 feature rows ────────────────────────────────────

# Scaler constants from feature_scaler_stats.csv
YEAR_MIN, YEAR_MAX = 2018.0, 2022.0
year_2023_norm = (2023 - YEAR_MIN) / (YEAR_MAX - YEAR_MIN)   # = 1.25

def month_to_sin_cos(m):
    """Convert integer month (1-12) to cyclic sin/cos encoding."""
    angle = 2 * np.pi * (m - 1) / 12
    return np.sin(angle), np.cos(angle)

rows_2023 = []
for _, row in zip_month_avgs.iterrows():
    z = row['zip']
    m = row['month_int']
    ms, mc = month_to_sin_cos(m)
    rows_2023.append({
        'zip'           : z,
        'month'         : m,
        'month_sin'     : ms,
        'month_cos'     : mc,
        'year'          : year_2023_norm,
        'avg_tmax_c'    : row['avg_tmax_c'],
        'temp_range'    : row['temp_range'],
        'tot_prcp_mm'   : row['tot_prcp_mm'],
        'dryness_3m_avg': row['dryness_3m_avg'],
        'latitude'      : row['latitude'],
        'longitude'     : row['longitude'],
    })

df_2023 = pd.DataFrame(rows_2023)
print(f'2023 synthetic rows: {len(df_2023):,}')
print(df_2023.head(3).to_string())

In [ ]:
# ── 8c. Predict with best model ───────────────────────────────────────────────
X_2023 = df_2023[FEATURES].values
y_pred_2023_log = best_model.predict(X_2023)
y_pred_2023_raw = np.expm1(y_pred_2023_log).clip(min=0)

df_2023['predicted_num_fires'] = y_pred_2023_raw

# Save output
out_cols = ['zip', 'month', 'predicted_num_fires']
out_path = DATA_DIR + 'classical_predictions_2023.csv'
df_2023[out_cols].to_csv(out_path, index=False)

print(f'Saved {len(df_2023):,} predictions to: {out_path}')
print(f'\nPrediction stats:')
print(df_2023['predicted_num_fires'].describe().round(4).to_string())

In [ ]:
# Distribution of 2023 predictions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_2023['predicted_num_fires'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of 2023 Predicted num_fires (all zips)')
axes[0].set_xlabel('Predicted num_fires')
axes[0].set_ylabel('Count')

# Monthly totals
monthly = df_2023.groupby('month')['predicted_num_fires'].sum().reset_index()
axes[1].bar(monthly['month'], monthly['predicted_num_fires'], color='#E24A33', edgecolor='white')
axes[1].set_title('Total Predicted Fires by Month — 2023')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Sum of predicted_num_fires')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])

plt.suptitle(f'2023 Predictions — {best_model_name}', fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Top 20 Highest-Risk Zip+Month Combos for 2023

In [ ]:
top20 = (
    df_2023[['zip', 'month', 'predicted_num_fires']]
    .sort_values('predicted_num_fires', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top20.index += 1   # 1-based rank
top20.index.name = 'Rank'

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
top20['month_name'] = top20['month'].map(month_names)
top20['predicted_num_fires'] = top20['predicted_num_fires'].round(4)

print('=== Top 20 Highest-Risk Zip+Month Combos for 2023 ===')
print(top20[['zip', 'month_name', 'predicted_num_fires']].to_string())

In [ ]:
# Horizontal bar chart of top 20
fig, ax = plt.subplots(figsize=(10, 7))

labels = [f"{row['zip']} — {row['month_name']}" for _, row in top20.iterrows()]
values = top20['predicted_num_fires'].values

bars = ax.barh(
    labels[::-1],   # highest at top
    values[::-1],
    color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, 20))
)
ax.set_xlabel('Predicted num_fires')
ax.set_title(f'Top 20 Highest-Risk Zip+Month — 2023 ({best_model_name})')

for bar, v in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 0.005 * values.max(),
            bar.get_y() + bar.get_height() / 2,
            f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Heat-map: top 20 zips across all 12 months
top20_zips = top20['zip'].unique()[:20]

pivot = (
    df_2023[df_2023['zip'].isin(top20_zips)]
    .pivot(index='zip', columns='month', values='predicted_num_fires')
    .fillna(0)
)
pivot.columns = [month_names[c] for c in pivot.columns]
# Sort rows by mean predicted fires
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Predicted num_fires'})
ax.set_title(f'Monthly Fire Risk Heat-Map — Top 20 Zips in 2023 ({best_model_name})')
ax.set_xlabel('Month')
ax.set_ylabel('Zip Code')
plt.tight_layout()
plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('='*60)
print('CLASSICAL BENCHMARK — FINAL SUMMARY')
print('='*60)
print(f'Best model : {best_model_name}')
print()
print('Validation (2021):')
for k, v in val_results[best_model_name].items():
    print(f'  {k:<14}: {v:.4f}')
print()
print('Test (2022):')
for k, v in test_metrics.items():
    print(f'  {k:<14}: {v:.4f}')
print()
print(f'2023 predictions saved to: {out_path}')
print(f'Total zip+month combos predicted: {len(df_2023):,}')
print(f'Highest predicted num_fires: {df_2023["predicted_num_fires"].max():.4f}')
print('='*60)